# 79. 时间线与甘特图（px.timeline）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 16 / 18 步：表达层级、流程、贡献与地域**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 瀑布图（Waterfall）  →  **本章任务：** 时间线与甘特图（px.timeline）  →  **下一步：** 地图图表（Map / Geo）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

项目推进时，光靠一长串任务清单很难看清谁先谁后、哪几项在并行推进。



## 本章目标

学完本章，你将能够：

- **理解**：理解「时间线与甘特图（px.timeline）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「时间线与甘特图（px.timeline）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「时间线与甘特图（px.timeline）」并读出其中的结论。


## 79.1 适用场景

**背景引入**：项目推进时，光靠一长串任务清单很难看清谁先谁后、哪几项在并行推进。用带开始和结束日期的横条把任务按时间排开，一眼就能看出整体进度、任务重叠和负责人分工。这一章用 Plotly 的时间线把散落的任务整理成直观的甘特图，面对几十个任务也能快速回答「现在进行到哪一步、接下来该做什么」。（好比健身房排课表：每个任务是一条横杠，左端是开始、右端是结束，横杠在时间轴上左右排，就能看出谁先谁后、哪几项同时在练、谁和谁有重叠。）

计划或复盘具有开始和结束日期的任务。


## 79.2 数据结构

任务名称、开始时间、结束时间和可选分组字段。


## 79.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 color 从 "owner" 改为 "phase" 或任务字段，观察颜色分组维度切换的效果
2. 修改 update_yaxes 的 autorange="reversed" 为默认，对比任务排列顺序的差异
3. 添加 hover_data 显示任务持续天数，说明悬浮信息对时间跨度读取的作用


## 79.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.timeline()`、`fig.update_yaxes()`、`fig.update_xaxes()`、`fig.update_layout()` | 计划或复盘具有开始和结束日期的任务。 | 结束时间早于开始时间 |
| 进阶变体 | `timeline.copy()`、`px.timeline()`、`fig.update_yaxes()`、`fig.update_xaxes()` | 在基础图表上增加分组、注释、布局或交互 | 任务顺序与执行顺序相反 |
| 关键参数 | `x_start/x_end` | 时间 | 结束时间早于开始时间 |
| 关键参数 | `y` | 任务 | 任务顺序与执行顺序相反 |
| 关键参数 | `color` | 负责人 | 时间线代替详细依赖管理 |
| 关键参数 | `category_orders` | 任务顺序 | 结束时间早于开始时间 |


## 79.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-79 -->
### 数学推导｜时间线中的持续时间

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜统一时间基准。** 把开始与结束时间都转换到同一时区，记为 $t_i^{start}$、$t_i^{end}$。

**第 2 步｜相减得到时间差。** $\Delta t_i=t_i^{end}-t_i^{start}$。

**第 3 步｜换成业务单位。** 若底层差值以秒计，则

$$
duration_i^{hour}=\frac{\Delta t_i^{second}}{3600}
$$

负持续时间说明顺序或时区有问题，不能通过取绝对值掩盖。

**把上面的关系收束为本章计算式：**

$$
duration_i=t_i^{end}-t_i^{start}
$$

**符号解释：** 开始与结束时间必须使用同一时区和时间单位。

**代码对应：** 解析时间后先计算持续时长，检查负值与重叠，再绘制甘特图。

**使用边界：** 缺失结束时间、跨时区和并行任务需要显式规则。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(f"Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行")


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 79.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.timeline(
    timeline,
    x_start="start",
    x_end="finish",
    y="task",
    color="owner",
    title="数据分析项目时间线",
)
fig.update_yaxes(autorange="reversed", title="任务")
fig.update_xaxes(title="日期")
fig.update_layout(legend_title="负责人")
fig.show()


**练一练**：基础图表的 `color="owner"` 会按负责人为每条任务配色。试着把颜色分组换成一个固定字符串（例如 `color="数据可视化实训"`），让所有任务共享同一种颜色，感受颜色编码从「分组」变成「单色」的区别。填好下面代码中的空缺，使 `fig_lian.data` 只有 1 组，并运行下方的自检确认。


In [ ]:
# 请在下方填写代码
# 目标：把时间线按「单一固定取值」的分组字段着色，使 fig_lian.data 只有 1 组。
# 提示：给 timeline 加一列固定取值的列 group="实训"，再把 color 指向它即可（color=<该列名>）。
import pandas as pd
import plotly.express as px


In [ ]:
# 完整答案：加一列固定取值的分组字段，color 指向它 → 单一分组
import pandas as pd
import plotly.express as px

tl = timeline.assign(group="实训")

fig_lian = px.timeline(
    tl,
    x_start="start",
    x_end="finish",
    y="task",
    color="group",  # 所有任务同属 "group"，颜色不分组
)


## 79.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
timeline_detail = timeline.copy()
timeline_detail["duration"] = (
    timeline_detail["finish"] - timeline_detail["start"]
).dt.days
fig = px.timeline(
    timeline_detail,
    x_start="start",
    x_end="finish",
    y="owner",
    color="task",
    hover_data=["duration"],
    title="按负责人查看项目安排",
)
fig.update_yaxes(autorange="reversed", title="负责人")
fig.update_xaxes(title="日期")
fig.update_layout(legend_title="任务")
fig.show()


## 79.8 参数说明

- x_start/x_end：时间
- y：任务
- color：负责人
- category_orders：任务顺序


## 79.9 结果解读

比较任务持续时间、依赖重叠和关键时间段。


## 79.10 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig = px.bar(report, x="region", y="sales", title="地区销售额")
fig.show()


### 79.10.1 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
fig.show()


### 79.10.2 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 79.11 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 79.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 79.12 易错点提醒

- 结束时间早于开始时间
- 任务顺序与执行顺序相反
- 时间线代替详细依赖管理


## 79.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 79.14 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：按负责人分面，观察各成员的排期
# 【目标】用 facet 按负责人拆开，看每人的任务排期。
import plotly.express as px

# 起点示例(已可运行)：加 facet_col，按负责人拆分时间线。
fig = px.timeline(
    timeline,
    x_start="start",
    x_end="finish",
    y="task",
    color="owner",
    facet_col="owner",
    title="分负责人项目时间线",
)
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(title="日期")
fig.update_layout(legend_title="负责人")
fig.show()

# ---- 反思记录：按人分面后，哪段时间最紧张 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
practice_timeline = pd.DataFrame(
    {
        "task": ["需求确认", "数据清洗", "模型分析", "汇报制作"],
        "start": pd.to_datetime(
            ["2026-04-01", "2026-04-03", "2026-04-06", "2026-04-10"]
        ),
        "finish": pd.to_datetime(
            ["2026-04-03", "2026-04-07", "2026-04-11", "2026-04-13"]
        ),
        "phase": ["准备", "准备", "分析", "交付"],
    }
)
fig = px.timeline(
    practice_timeline,
    x_start="start",
    x_end="finish",
    y="task",
    color="phase",
    title="分析任务计划",
)
fig.update_yaxes(autorange="reversed")
fig.show()


## 79.15 小结

用时间线展示任务起止、重叠、负责人和项目节奏。


### 79.15.1 你已经掌握

- 判断时间线与甘特图（px.timeline）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 79.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `x_start/x_end` | 时间 |
| `y` | 任务 |
| `color` | 负责人 |
| `category_orders` | 任务顺序 |


### 79.15.3 需要注意

- 结束时间早于开始时间
- 任务顺序与执行顺序相反
- 时间线代替详细依赖管理


### 79.15.4 完成检查

- [ ] 能判断什么问题适合使用时间线与甘特图（px.timeline）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 79.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
